# BC K-12 Education — Funding vs Enrolment (SQL / SQLite)

Self-directed analysis on public BC open data. Two government datasets pulled
from source, cleaned in Python, then analysed in SQL: does school funding track
student enrolment across districts?

**Sources** (Open Government Licence – British Columbia):
- Funding — BC Ministry of Education & Child Care, *Summary of Grants to Date*
- Enrolment — *BC Schools: Student Enrolment and FTE*, open.canada.ca

**Result:** Funding +20.9%, FTE +3.2% (2022/23→2025/26); Pearson r = 0.91.
Every district that lost students still gained funding.

In [2]:
import pandas as pd
import sqlite3

folder = '/Users/yancizy/Downloads/bc-education-sql/Clean_data/'

# --- Read source files ---
df = pd.read_excel(folder + 'bc_funding_4yr_by_district.xlsx', skiprows=3)
e  = pd.read_excel(folder + 'student_enrolment_and_fte_2020-21_to_2025_26.xlsx')

# --- Clean enrolment: 137,581 raw rows -> 360 district-year rows ---
# Filter to avoid double-counting students.
e = e[(e['DATA_LEVEL'] == 'District Level') &
      (e['PUBLIC_OR_INDEPENDENT'] == 'Public School') &
      (e['GRADE'] == 'All Grades') &
      (e['FACILITY_TYPE'] == 'All Facility Types')]

# --- Align JOIN keys across the two datasets ---
df['Year'] = df['Year'].str.replace('-', '/')                       # 2022-23   -> 2022/23
e['SCHOOL_YEAR'] = e['SCHOOL_YEAR'].str[:5] + e['SCHOOL_YEAR'].str[7:]  # 2022/2023 -> 2022/23
df['District #'] = df['District #'].astype(int)                     # ensure int key
e = e.dropna(subset=['DISTRICT_NUMBER'])
e['DISTRICT_NUMBER'] = e['DISTRICT_NUMBER'].astype(int)            # 5.0 -> 5

print('funding:', df.shape, '| enrolment:', e.shape)

funding: (240, 14) | enrolment: (360, 23)


In [3]:
# Load both clean frames into an in-memory SQLite database.
# From here on, all analysis is done in SQL.
con = sqlite3.connect(':memory:')
df.to_sql('funding', con, index=False, if_exists='replace')
e.to_sql('enrolment', con, index=False, if_exists='replace')

print(pd.read_sql('SELECT COUNT(*) AS funding_rows FROM funding', con))
print(pd.read_sql('SELECT COUNT(*) AS enrolment_rows FROM enrolment', con))

   funding_rows
0           240
   enrolment_rows
0             360


In [4]:
# Q1 — Total funding by year (province-wide)
query = """
SELECT
    Year,
    ROUND(SUM("Total District Funding ($)") / 1e9, 3) AS funding_billions
FROM funding
GROUP BY Year
ORDER BY Year
"""
print(pd.read_sql(query, con))

      Year  funding_billions
0  2022/23             7.063
1  2023/24             7.820
2  2024/25             8.251
3  2025/26             8.537


In [5]:
# Q2 — Correlation between district funding change and FTE change (2022/23 -> 2025/26)
# Joins funding + enrolment per district, then correlates the two 4-year % changes.
# Note: SQLite does integer division on integer columns, so funding is multiplied
# by 1.0 to force float division. CORR() is not in SQLite, so r is computed in pandas.
query = """
WITH f AS (
    SELECT "District #" AS district_number,
        SUM(CASE WHEN Year='2022/23' THEN "Total District Funding ($)" END) AS f_start,
        SUM(CASE WHEN Year='2025/26' THEN "Total District Funding ($)" END) AS f_end
    FROM funding GROUP BY "District #"
),
en AS (
    SELECT DISTRICT_NUMBER AS district_number,
        SUM(CASE WHEN SCHOOL_YEAR='2022/23' THEN TOTAL_FTE END) AS e_start,
        SUM(CASE WHEN SCHOOL_YEAR='2025/26' THEN TOTAL_FTE END) AS e_end
    FROM enrolment GROUP BY DISTRICT_NUMBER
)
SELECT
    (f.f_end * 1.0 / f.f_start - 1.0)   AS funding_change,
    (en.e_end * 1.0 / en.e_start - 1.0) AS fte_change
FROM f
JOIN en ON f.district_number = en.district_number
WHERE f.f_start > 0 AND en.e_start > 0
"""
changes = pd.read_sql(query, con)
r = changes['funding_change'].corr(changes['fte_change'])

print('Districts joined:', len(changes))
print('Pearson r =', round(r, 2))

Districts joined: 60
Pearson r = 0.91
